# Granularity Experiment

Research question: at what n_classes does performance peak before declining?

Notebook flow:
1. Oversampling comparison (binary) — pick best sampler
2. TF-IDF cosine similarity intra-game — identify natural class merge candidates
3. Generate src/label_schemes.py from empirical findings
4. Incremental granularity experiment — all n_classes, all models, full metrics
5. Sweet spot plots

## 0. Imports & CONFIG

In [ ]:
# Standard imports — consistent with project style
import warnings, time, json, textwrap
from pathlib import Path
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.base import clone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
from imblearn.over_sampling import RandomOverSampler, SMOTE, BorderlineSMOTE, ADASYN
import sys
sys.path.insert(0, str(Path('..').resolve()))
from src.loaders import load_wot, load_dota
from src.pipelines import build_pipe
from src.scoring import cv_score, append_registry

CONFIG = {
    'seed': 7524,
    'cv_folds': 5,
    'text_col': 'clean_message',
    'label_col': 'label',
    'registry_path': Path('../data/results/results_registry.csv'),
    'label_schemes_path': Path('../src/label_schemes.py'),
}
seed = CONFIG['seed']
cv   = StratifiedKFold(n_splits=CONFIG['cv_folds'], shuffle=True, random_state=seed)
np.random.seed(seed)
WOT_CLASSES  = {0:'Non-Toxic', 1:'Insults', 2:'Other Offensive', 3:'Hate', 4:'Threats', 5:'Extremism'}
DOTA_CLASSES = {0:'Non-Toxic', 1:'Ego', 2:'Aggression', 3:'Impolite'}
print('CONFIG loaded.')

This notebook answers the core research question: **at what granularity does per-game classifier performance peak?**

**Outputs produced:**
- `src/label_schemes.py` — empirically derived merge dicts for all granularity levels
- `data/results/granularity_sweet_spot.png` — multi-metric line plot
- `data/results/intra_game_cosine_similarity.png` — class vocabulary overlap heatmap
- Rows appended to `data/results/results_registry.csv`

## 1. Oversampling Comparison (Binary)

In [ ]:
# Compare all 4 oversamplers at binary level to pick the best before running
# the full multiclass experiment. Binary is fastest — minimises wall time here.
wot_train_raw  = load_wot('train')
wot_val_raw    = load_wot('val')
dota_train_raw = load_dota('train')
dota_val_raw   = load_dota('val')

# Binarise for oversampler comparison: label > 0 → 1
X_wot_train  = wot_train_raw[CONFIG['text_col']]
y_wot_bin    = (wot_train_raw[CONFIG['label_col']].astype(int) > 0).astype(int)
X_dota_train = dota_train_raw[CONFIG['text_col']]
y_dota_bin   = (dota_train_raw[CONFIG['label_col']].astype(int) > 0).astype(int)

OVERSAMPLERS = {
    'RandomOverSampler': RandomOverSampler(random_state=seed),
    'SMOTE':             SMOTE(random_state=seed),
    'BorderlineSMOTE':   BorderlineSMOTE(random_state=seed),
    'ADASYN':            ADASYN(random_state=seed),
}

ref_clf   = LogisticRegression(C=1.0, max_iter=1000, random_state=seed, n_jobs=1)
os_results = []

for game, X_tr, y_tr in [('WoT', X_wot_train, y_wot_bin),
                           ('Dota', X_dota_train, y_dota_bin)]:
    print(f'=== {game} ===')
    for os_name, sampler in OVERSAMPLERS.items():
        pipe   = build_pipe(clone(ref_clf), oversampler=sampler)
        scores = cv_score(pipe, X_tr, y_tr, cv=cv)
        print(f'  {os_name:<20} macro_f1={scores["cv_macro_f1"]:.4f} '
              f'recall={scores["cv_recall_macro"]:.4f} '
              f'precision={scores["cv_precision_macro"]:.4f}')
        os_results.append({'game': game, 'oversampler': os_name, **scores})

os_df = pd.DataFrame(os_results)
print('\n--- Best per game ---')
for game in ['WoT', 'Dota']:
    best = os_df[os_df['game']==game].sort_values('cv_macro_f1', ascending=False).iloc[0]
    print(f'{game}: {best["oversampler"]} (macro_f1={best["cv_macro_f1"]:.4f})')

BEST_OS = {
    game: os_df[os_df['game']==game].sort_values('cv_macro_f1', ascending=False).iloc[0]['oversampler']
    for game in ['WoT', 'Dota']
}
OS_CLASS_MAP = {
    'RandomOverSampler': RandomOverSampler,
    'SMOTE':             SMOTE,
    'BorderlineSMOTE':   BorderlineSMOTE,
    'ADASYN':            ADASYN,
}
print(f'\nSelected: {BEST_OS}')

## 1. Oversampling — Interpretation

Oversampler choice matters because gaming toxicity data is heavily skewed (Non-Toxic majority).
A sampler that boosts rare-class recall without hurting precision on the majority class wins.

- **RandomOverSampler**: duplicates minority samples — fast, no interpolation risk
- **SMOTE / BorderlineSMOTE**: interpolates new synthetic points along decision boundary
- **ADASYN**: focuses synthesis on hard-to-classify borderline samples

The winner here is used for **all** subsequent granularity experiments to keep comparisons fair.
If SMOTE or ADASYN wins, that signals the minority class is separable enough for interpolation to help.

## 2. TF-IDF Cosine Similarity — Intra-Game Class Clustering

In [ ]:
# Compute TF-IDF centroid per class and cosine similarity matrix for each game.
# High similarity between two classes = they share vocabulary = merge candidates.
# This drives the label scheme design — not assumptions.

def class_centroids(texts: pd.Series, labels: pd.Series, n_classes: int) -> np.ndarray:
    """Fit TF-IDF on texts, return L2-normalised centroid matrix (n_classes × vocab)."""
    tfidf = TfidfVectorizer(ngram_range=(1,2), min_df=3, max_df=0.95,
                             sublinear_tf=True, norm='l2')
    X = tfidf.fit_transform(texts)
    centroids = []
    for c in range(n_classes):
        mask = (labels.astype(int) == c).values
        centroids.append(np.asarray(X[mask].mean(axis=0)))
    return normalize(np.vstack(centroids))

wot_centroids  = class_centroids(wot_train_raw[CONFIG['text_col']],
                                   wot_train_raw[CONFIG['label_col']], 6)
dota_centroids = class_centroids(dota_train_raw[CONFIG['text_col']],
                                   dota_train_raw[CONFIG['label_col']], 4)

wot_sim  = cosine_similarity(wot_centroids)
dota_sim = cosine_similarity(dota_centroids)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

wot_names  = list(WOT_CLASSES.values())
dota_names = list(DOTA_CLASSES.values())

for ax, sim, names, title in [
    (axes[0], wot_sim,  wot_names,  'WoT — Class Cosine Similarity'),
    (axes[1], dota_sim, dota_names, 'Dota — Class Cosine Similarity'),
]:
    mask = np.eye(len(names), dtype=bool)   # hide diagonal (self-similarity = 1.0)
    sim_display = sim.copy()
    sim_display[mask] = np.nan
    sns.heatmap(sim_display, annot=True, fmt='.3f', cmap='YlOrRd',
                xticklabels=names, yticklabels=names, ax=ax, vmin=0, vmax=1)
    ax.set_title(title, fontweight='bold', fontsize=12)
    ax.tick_params(axis='x', rotation=30, labelsize=9)
    ax.tick_params(axis='y', rotation=0, labelsize=9)

plt.suptitle('Intra-Game Class Vocabulary Similarity', fontweight='bold', fontsize=14)
plt.tight_layout()
Path('../data/results').mkdir(parents=True, exist_ok=True)
plt.savefig('../data/results/intra_game_cosine_similarity.png', dpi=150, bbox_inches='tight')
plt.show()

# Print ranked merge candidates
print('\n=== WoT merge candidates (sorted by similarity, excluding diagonal) ===')
for i in range(6):
    for j in range(i+1, 6):
        print(f'  {wot_names[i]} \u2194 {wot_names[j]}: {wot_sim[i,j]:.3f}')

print('\n=== Dota merge candidates ===')
for i in range(4):
    for j in range(i+1, 4):
        print(f'  {dota_names[i]} \u2194 {dota_names[j]}: {dota_sim[i,j]:.3f}')

## 2. Cosine Similarity — Interpretation

High similarity (> 0.X) between two classes indicates shared vocabulary —
these are natural merge candidates for coarser granularity levels.

**Read the heatmap and define merges below based on the actual values.**
Classes with lowest similarity to all others are hardest to merge without information loss.

## 3. Define Label Schemes from Empirical Findings

In [ ]:
# Define merge dicts based on the cosine similarity heatmap above.
# Edit these dicts to reflect what the heatmap reveals about natural clusters.
# HIGH similarity pairs → merge first. LOW similarity pairs → keep separate longest.
#
# Format: {original_label: new_label, ...}
# New labels must be 0-indexed and contiguous (0, 1, 2, ...)

# ── WoT schemes ──────────────────────────────────────────────────────────────
# Edit based on heatmap findings:
WOT_SCHEME_2 = {0: 0, 1: 1, 2: 1, 3: 1, 4: 1, 5: 1}          # binary (fixed)
WOT_SCHEME_3 = {0: 0, 1: 1, 2: 1, 3: 1, 4: 2, 5: 2}          # PLACEHOLDER — edit from heatmap
WOT_SCHEME_4 = {0: 0, 1: 1, 2: 1, 3: 2, 4: 3, 5: 3}          # PLACEHOLDER — edit from heatmap
WOT_SCHEME_5 = {0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 4}          # PLACEHOLDER — edit from heatmap
WOT_SCHEME_6 = {0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5}          # identity (fixed)

# ── Dota schemes ─────────────────────────────────────────────────────────────
DOTA_SCHEME_2 = {0: 0, 1: 1, 2: 1, 3: 1}                      # binary (fixed)
DOTA_SCHEME_3 = {0: 0, 1: 1, 2: 1, 3: 2}                      # PLACEHOLDER — edit from heatmap
DOTA_SCHEME_4 = {0: 0, 1: 1, 2: 2, 3: 3}                      # identity (fixed)

# ── Class name labels for display ─────────────────────────────────────────────
WOT_CLASS_NAMES = {
    2: ['Non-Toxic', 'Toxic'],
    3: ['Non-Toxic', 'Group-A', 'Group-B'],          # update names to match your merges
    4: ['Non-Toxic', 'Group-A', 'Group-B', 'Group-C'],
    5: ['Non-Toxic', 'Insults', 'Other Off.', 'Hate', 'Threats+Extremism'],
    6: list(WOT_CLASSES.values()),
}
DOTA_CLASS_NAMES = {
    2: ['Non-Toxic', 'Toxic'],
    3: ['Non-Toxic', 'Group-A', 'Group-B'],           # update names to match your merges
    4: list(DOTA_CLASSES.values()),
}

WOT_SCHEMES  = {2: WOT_SCHEME_2,  3: WOT_SCHEME_3,  4: WOT_SCHEME_4,
                5: WOT_SCHEME_5,  6: WOT_SCHEME_6}
DOTA_SCHEMES = {2: DOTA_SCHEME_2, 3: DOTA_SCHEME_3, 4: DOTA_SCHEME_4}

print('Schemes defined. Edit dicts above based on heatmap before proceeding.')
print('WoT schemes:', list(WOT_SCHEMES.keys()))
print('Dota schemes:', list(DOTA_SCHEMES.keys()))

## 3. Label Schemes — Edit Before Proceeding

The placeholder dicts above must be edited to match heatmap findings.
Binary (n=2) and identity (n=6 WoT, n=4 Dota) are fixed.
Intermediate schemes (3, 4, 5 for WoT; 3 for Dota) should reflect the
most similar class pairs identified in the heatmap.

Once edited, the next cell writes `src/label_schemes.py`.

## 4. Write src/label_schemes.py

In [ ]:
# Generate src/label_schemes.py as a product of this notebook's analysis.
# This file is the empirical output — not a hardcoded assumption.
label_schemes_content = textwrap.dedent(f"""\
    # Auto-generated by notebooks/01_granularity_experiment.ipynb
    # Based on TF-IDF cosine similarity analysis — edit Section 3 of that notebook to update.

    WOT_SCHEMES = {WOT_SCHEMES!r}

    DOTA_SCHEMES = {DOTA_SCHEMES!r}

    WOT_CLASS_NAMES = {WOT_CLASS_NAMES!r}

    DOTA_CLASS_NAMES = {DOTA_CLASS_NAMES!r}
    """)

CONFIG['label_schemes_path'].write_text(label_schemes_content, encoding='utf-8')
print(f'Written: {CONFIG["label_schemes_path"]}')
print('Contents preview:')
print(label_schemes_content[:400])

`src/label_schemes.py` is now importable by notebooks 02, 03, 04, 06.
Re-run this cell whenever you update the scheme dicts in Section 3.

## 5. Incremental Granularity Experiment

In [ ]:
# Run all 3 models at every granularity level for each game.
# No minimum class size gate — test every scenario.
# Use best oversampler from Section 1.
from src.label_schemes import (WOT_SCHEMES as WS, DOTA_SCHEMES as DS,
                                WOT_CLASS_NAMES as WCN, DOTA_CLASS_NAMES as DCN)

def run_granularity_track(game: str, raw_train: pd.DataFrame, schemes: dict,
                           class_names_map: dict, best_os_name: str,
                           registry_path: Path):
    results = []
    OsClass = OS_CLASS_MAP[best_os_name]

    for n_classes, scheme in schemes.items():
        # Apply scheme inline — no loader scheme param needed
        train = raw_train.copy()
        train[CONFIG['label_col']] = train[CONFIG['label_col']].astype(int).map(scheme)
        X_train = train[CONFIG['text_col']]
        y_train = train[CONFIG['label_col']]

        classifiers = {
            'Logistic Regression': LogisticRegression(C=1.0, max_iter=1000, random_state=seed, n_jobs=1),
            'Naive Bayes':         MultinomialNB(),
            'LinearSVC':           LinearSVC(C=1.0, max_iter=2000, tol=1e-3, random_state=seed),
        }

        best_f1, best_name = -1, None
        for clf_name, clf in classifiers.items():
            pipe   = build_pipe(clone(clf), oversampler=OsClass(random_state=seed))
            scores = cv_score(pipe, X_train, y_train, cv=cv)
            print(f'  {game} n={n_classes} {clf_name:<22} '
                  f'f1={scores["cv_macro_f1"]:.4f} '
                  f'rec={scores["cv_recall_macro"]:.4f} '
                  f'prec={scores["cv_precision_macro"]:.4f}')
            if scores['cv_macro_f1'] > best_f1:
                best_f1, best_name = scores['cv_macro_f1'], clf_name

            append_registry({
                'experiment': 'granularity_per_dataset',
                'train_game': game, 'test_game': game,
                'n_classes': n_classes, 'label_scheme': 'empirical',
                'model': clf_name,
                **scores,
                'test_macro_f1': None, 'test_weighted_f1': None,
                'test_precision_macro': None, 'test_recall_macro': None, 'test_auc': None,
                'per_class_recall': None,
                'ood_macro_f1': None, 'ood_weighted_f1': None,
                'ood_precision_macro': None, 'ood_recall_macro': None, 'ood_auc': None,
                'anomaly_auroc': None,
                'notes': '',
            }, path=registry_path)

        results.append({'game': game, 'n_classes': n_classes,
                        'best_model': best_name, 'best_cv_f1': round(best_f1, 4)})
        print(f'  -> best: {best_name} ({best_f1:.4f})\n')

    return pd.DataFrame(results)

print('=== WoT incremental ===')
wot_results = run_granularity_track('WoT', wot_train_raw, WS, WCN,
                                     BEST_OS['WoT'], CONFIG['registry_path'])
print('\n=== Dota incremental ===')
dota_results = run_granularity_track('Dota', dota_train_raw, DS, DCN,
                                      BEST_OS['Dota'], CONFIG['registry_path'])

print('\nWoT:')
print(wot_results.to_string(index=False))
print('\nDota:')
print(dota_results.to_string(index=False))

## 5. Incremental Results — Interpretation

Each row = best CV macro F1 across all three classifiers at that granularity.

**Sweet spot** = the n_classes where macro F1 peaks before declining.
Declining F1 at higher n means the classes are too vocabulary-similar for TF-IDF to discriminate —
a signal to merge them (already captured in the label schemes above).

## 6. Sweet Spot Plots

In [ ]:
# Multi-metric line plots: F1, recall, precision per game.
# Sweet spot = where metrics peak before declining with more classes.
registry = pd.read_csv(CONFIG['registry_path'])
gran = registry[registry['experiment']=='granularity_per_dataset']

metrics = [('cv_macro_f1',      'Macro F1',  '#1565C0'),
           ('cv_recall_macro',   'Recall',    '#C62828'),
           ('cv_precision_macro','Precision', '#2E7D32')]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, game in zip(axes, ['WoT', 'Dota']):
    game_df    = gran[gran['train_game']==game]
    best_per_n = (game_df.groupby('n_classes')[['cv_macro_f1','cv_recall_macro','cv_precision_macro']]
                         .max().reset_index())
    for col, label, color in metrics:
        ax.plot(best_per_n['n_classes'], best_per_n[col], marker='o',
                linewidth=2, color=color, label=label)
    ax.set_xlabel('Number of Classes', fontsize=12)
    ax.set_ylabel('Score', fontsize=12)
    ax.set_title(f'{game} — Metrics vs Granularity', fontweight='bold')
    ax.legend()
    ax.set_ylim(0.4, 1.0)
    ax.grid(True, alpha=0.3)

plt.suptitle('Granularity Sweet Spot', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('../data/results/granularity_sweet_spot.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Sweet Spot — Interpretation

The sweet spot is the n_classes where **all three metrics (F1, recall, precision) peak together**
before diverging or declining.

- If F1 and recall peak at n=3 but precision keeps rising → classes are distinct but imbalanced
- If all three peak at n=2 → fine-grained labels add noise, binary is the ceiling for this feature set
- Downstream notebooks (03 holdout, 04 OOD) use the sweet-spot n_classes as the default target